In [12]:
from pathlib import Path

from typing import Any

from matplotlib.backend_bases import MouseEvent

from bob.core import (
    p223,
    Device,
    get_datagraph,
    bind_model_namespace,
    dump,
)

from bob.devices.hvac.damper import ElectricalActuatedDamper
from bob.devices.hvac.coil import ChilledWaterCoil, HotWaterCoil
from bob.devices.hvac.fan import Fan
from bob.devices.hvac.filter import Filter
from bob.devices.hvac.damper import Window
from bob.devices.hvac.boiler import HotWaterBoiler, ElectricalHotWaterBoiler
from bob.devices.hvac.valve import WaterValve
from bob.devices.lighting.light import Light
from bob.systems.hvac.airhandlingunit import AirHandlingUnit
from bob.systems.hvac.vav import VAV
from bob.devices.hvac.pump import Pump
from bob.sensor.temperature import AirTemperatureSensor
from bob.sensor.flow import AirFlowSensor
from bob.sensor.movement import MovementSensor

from bob.space.physical import Building, Floor, Roof, Office, Room, Bathroom, Corridor
from bob.space.hvac import HVACSpace, HVACZone
from bob.space.light import LightingSpace, LightingZone

from bob.connections.air import *
from bob.connections.water import WaterConnection, WaterInletConnectionPoint, WaterOutletConnectionPoint

__namespace__ = p223

class AgnosticWaterBoiler(Device):
    node_type = p223.AgnosticBoiler
    waterInlet: WaterInletConnectionPoint
    waterOutlet: WaterOutletConnectionPoint

class AgnosticWaterCoil(Device):
    node_type = p223.AgnosticCoil
    waterInlet: WaterInletConnectionPoint
    waterOutlet: WaterOutletConnectionPoint

class HotWaterTank(Device):
    node_type = p223.HotWaterTank
    waterInlet: WaterInletConnectionPoint
    waterOutlet: WaterOutletConnectionPoint

DHWBoiler = AgnosticWaterBoiler(label='DHW-BOILER')
htgloop_boiler = AgnosticWaterBoiler(label='HTGLOOP-BOILER')
dhw_hot_water_tank = HotWaterTank(label='DHW-TANK')

htg_hot_water_tank = HotWaterTank(label='HTG-TANK')

city_water_tap = WaterConnection(label='CITY-WATER')
house_faucets = WaterConnection(label='HouseFaucets')
kitchen_faucet = WaterValve(label='KITCHENFAUCET')
house_drain = WaterConnection(label='HOUSE-DRAIN')
city_drain = WaterConnection(label='CityDrain')

htg_pump = Pump(label='HowWaterPump')
house_hw_supply = WaterConnection(label='HotWaterSupply')
house_hw_return = WaterConnection(label='HotWaterReturn')
joelsofficeheatingvalve = WaterValve(label='joelsofficehtgvlv')
joelsofficeheatingcoil = AgnosticWaterCoil(label='JoelsOfficeCoil')
fillingValve = WaterValve(label='FILL-VLV', comment='Fill Hot Water Loop with water')
city_water_tap >> DHWBoiler.waterInlet
DHWBoiler.waterOutlet >> dhw_hot_water_tank.waterInlet
dhw_hot_water_tank.waterOutlet >> house_faucets >> kitchen_faucet.waterInlet
kitchen_faucet.waterOutlet >> house_drain >> city_drain

htg_hot_water_tank.waterOutlet >> htgloop_boiler.waterInlet
htgloop_boiler.waterOutlet >> htg_pump.waterInlet
htg_pump.waterOutlet >> house_hw_supply >> joelsofficeheatingvalve.waterInlet
joelsofficeheatingvalve.waterOutlet >> joelsofficeheatingcoil.waterInlet
joelsofficeheatingcoil.waterOutlet >> house_hw_return
house_hw_return >> htg_hot_water_tank.waterInlet
city_water_tap >> fillingValve.waterInlet
fillingValve.waterOutlet >> house_hw_return


RuntimeError: no common connection types